# 第 02 章：结构化输出与规范 (Pydantic)

> **学习定位**：本章属于「第一阶段：把 LangChain 当开发入口」。
> 这一章的 notebook 目标很明确：把 `with_structured_output()` 跑起来，亲眼看到模型输出如何从普通文本变成可验证的 Python 对象。

本章实验会按这个顺序推进：
1. 初始化模型
2. 定义最小 Schema
3. 用 `with_structured_output()` 绑定结构化能力
4. 验证返回值确实是对象
5. 验证嵌套 Schema 也能工作
6. 观察边界情况


## 1. 模型初始化

这里我们继续使用第 01 章的统一入口 `init_chat_model`。

In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field

load_dotenv()

llm = init_chat_model(
    model="deepseek-chat",
    model_provider="deepseek",
    base_url="https://api.deepseek.com",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
)

print(f"模型加载成功：{llm.__class__.__name__}")


模型加载成功：ChatDeepSeek


## 2. 定义一个最小 Schema

先不要追求复杂，先用一个最小对象观察结构化返回。

In [2]:
class UserProfile(BaseModel):
    name: str = Field(..., description="用户的全名")
    age: int = Field(..., description="用户的年龄")
    interests: list[str] = Field(
        default_factory=list,
        description="用户的兴趣爱好列表",
    )

print(UserProfile.model_json_schema())


{'properties': {'name': {'description': '用户的全名', 'title': 'Name', 'type': 'string'}, 'age': {'description': '用户的年龄', 'title': 'Age', 'type': 'integer'}, 'interests': {'description': '用户的兴趣爱好列表', 'items': {'type': 'string'}, 'title': 'Interests', 'type': 'array'}}, 'required': ['name', 'age'], 'title': 'UserProfile', 'type': 'object'}


## 3. 使用 `with_structured_output()` 增强 LLM

注意这里我们增强的仍然是 `llm` 本身，不是 Agent。

In [3]:
structured_llm = llm.with_structured_output(UserProfile)

query = "我叫张三，今年25岁，平时喜欢打篮球、游泳，最近还在打黑神话悟空。"
result = structured_llm.invoke(query)

print("返回值类型:", type(result))
print(result)


返回值类型: <class '__main__.UserProfile'>
name='张三' age=25 interests=['打篮球', '游泳', '打黑神话悟空']


## 4. 直接访问对象字段

如果结构化输出成功，你现在拿到的就不再是字符串，而是可以直接点字段的对象。

In [4]:
print(f"姓名: {result.name}")
print(f"年龄: {result.age}")
print(f"兴趣: {result.interests}")


姓名: 张三
年龄: 25
兴趣: ['打篮球', '游泳', '打黑神话悟空']


## 5. 验证嵌套 Schema

真实业务里经常是一棵对象树，而不是平面三字段。

In [5]:
class DeliveryAddress(BaseModel):
    province: str = Field(description="省份，无需后缀'省'")
    city: str = Field(description="城市，无需后缀'市'")
    detail: str = Field(description="详细地址")

class OrderReceipt(BaseModel):
    customer_name: str = Field(description="客户真实姓名")
    shipping_address: DeliveryAddress = Field(description="收件地址")

order_llm = llm.with_structured_output(OrderReceipt)
text = "我是老李，你要安排发件到这边：四川的成都市武侯区科华北路66号。"
receipt = order_llm.invoke(text)

print("返回值类型:", type(receipt))
print(receipt)


返回值类型: <class '__main__.OrderReceipt'>
customer_name='老李' shipping_address=DeliveryAddress(province='四川', city='成都', detail='武侯区科华北路66号')


## 6. 访问嵌套字段

重点观察内层字段是否能稳定拿到。

In [6]:
print(f"客户姓名: {receipt.customer_name}")
print(f"收件城市: {receipt.shipping_address.city}")
print(f"详细地址: {receipt.shipping_address.detail}")


客户姓名: 老李
收件城市: 成都
详细地址: 武侯区科华北路66号


## 7. 边界实验

这里故意给一个信息不充分的输入，观察模型在结构化约束下会怎么表现。

In [7]:
ambiguous_query = "我叫小王，喜欢旅行。"
ambiguous_result = structured_llm.invoke(ambiguous_query)
print(type(ambiguous_result))
print(ambiguous_result)


<class '__main__.UserProfile'>
name='小王' age=0 interests=['旅行']


## 8. 小结

到这里你应该已经能区分三件事：

1. 原始 `llm.invoke(...)` 更像自然语言问答。
2. `llm.with_structured_output(...)` 是在增强模型输出契约。
3. 拿到结构化对象后，你的程序就能像处理普通 Python 对象一样继续往下走。
